In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
import warnings
warnings.filterwarnings('ignore')
from transformers import AutoTokenizer,AutoConfig, AutoModel, BitsAndBytesConfig

model_name_or_path ='zai-org/chatglm3-6b-base'
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    llm_inta_threshold = 6.0,
    llm_int8_has_fp16_weight = False
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, 
    trust_remote_code = True
)

model= AutoModel.from_pretrained(
    model_name_or_path,
    quantization_config = bnb_config,
    trust_remote_code = True
)

ImportError: The installed version of bitsandbytes (<0.43.1) requires CUDA, but CUDA is not available. You may need to install PyTorch with CUDA support or upgrade bitsandbytes to >=0.43.1.

In [33]:
import openai
# 新老 API 都会用到：openai.api_key 会被 get_completion 读取
openai.api_key = "sk-vx435cXksjsOMrs9aRtt00RyHSOwM0mMIlwHQVtOU6JTzCO5"

In [34]:
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

# 仅当 .env 里有 OPENAI_API_KEY 时才覆盖，避免把上面 cell 里设的 key 清空
if os.getenv("OPENAI_API_KEY"):
    openai.api_key = os.getenv("OPENAI_API_KEY")


In [35]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())  # 若用 .env，确保已加载

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    api_key = os.getenv("OPENAI_API_KEY") or getattr(openai, "api_key", None)
    if not api_key:
        raise ValueError(
            "请先运行上面「设置 openai.api_key」的 cell，或设置环境变量 OPENAI_API_KEY（可在项目根目录 .env 里写 OPENAI_API_KEY=sk-...）"
        )
    try:
        # openai >= 1.0.0
        client = openai.OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model, messages=messages, temperature=0
        )
        return response.choices[0].message.content
    except AttributeError:
        # openai < 1.0 (e.g. 0.28)
        response = openai.ChatCompletion.create(
            model=model, messages=messages, temperature=0
        )
        return response.choices[0].message["content"]

In [36]:
text = f"""
You should express what you want a model to do by \ 
providing instructions that are as clear and \ 
specific as you can possibly make them. \ 
This will guide the model towards the desired output, \ 
and reduce the chances of receiving irrelevant \ 
or incorrect responses. Don't confuse writing a \ 
clear prompt with writing a short prompt. \ 
In many cases, longer prompts provide more clarity \ 
and context for the model, which can lead to \ 
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \ 
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

APITimeoutError: Request timed out.